In [1]:
using Revise
using InteractiveUtils

const PATH_METADATA_ENRICH_JL = "functions/enrich_metadata.jl"
# const PATH_PDF_EXTRACT_JL = "functions/qog_pdf_extract.jl"
# const PATH_METADATA_ENHANCE_JL = "functions/qog_metadata_join.jl"

# Print a summary of all dataframes that are current loaded in Main
function dataframe_summaries(mod=Main)
    for n in names(mod)
        x = getfield(mod, n)
        if x isa AbstractDataFrame
            println("=== DataFrame: ", n, " ===")
            println(summary(x))
            println()
        end
    end
end

includet(PATH_METADATA_ENRICH_JL)

QoG Time-Series Loader Pipeline
Input: data/qog_std_ts_jan25.arrow

>>> Step 1/5: Loading raw data with identity promotion...
    Loaded: 12391 rows × 2010 columns
    ✓ ggis_rowid assigned

>>> Step 2/5: Previewing rescue collisions...
>>> RESCUE COLLISION PREVIEW (informational — NO rows will be deleted):
    Total (ccode, year) pairs with >1 row: 22

    By entity combination:
      VDR + VNM: 22 years (1955-1976)
        Years: 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976

    NOTE: Use `ggis_rowid` as unique key, or (ident_ccode, ident_year, ident_ccodealp)
    ⚠️  Found 1 collision group(s)
    (See year details above)

>>> Step 3/5: Rescuing historical ccodes...
>>> Historical Ccode Rescue:
    Missing before: 234
    Rescued: 234
    Missing after: 0
    Row count: 12391 (unchanged)
    By alpha code:
      ETH → 231: 47 rows
      YEM → 887: 44 rows
      DEU → 276: 42 rows
      MHL → 584: 3

In [2]:

dataframe_summaries()

=== DataFrame: REGION_LABELS ===
10×2 DataFrame

=== DataFrame: df ===
12391×2013 DataFrame

=== DataFrame: meta_df ===
2010×8 DataFrame



In [3]:
meta_plus1 = enrich_metadata_with_lifespan();


>>> Results Summary
    Variables Audited: 2009
      :modern — 887
      :experimental — 355
      :legacy — 240
      :current — 240
      :historical — 168
      :anchor — 119


In [4]:
check_year_discrepancies(meta_plus1)

No year discrepancies found exceeding tolerance 1.


In [9]:

meta = meta_plus1
# Pre-compute the universe as the script does
regional_universe = compute_regional_country_universe(df);

In [7]:
"""
    inspect_slug_geo(df, meta_df, slug, regional_universe)

Inspects the geographic data availability for a specific slug at its 
recorded death year. 'slug' can be a String or a Symbol.
"""
function inspect_slug_geo(df, meta_df, slug, regional_universe)
    # Convert slug to both formats for consistency
    slug_text = string(slug)
    slug_sym = Symbol(slug)

    # 1. Identify the target year being used for this slug
    target_row = subset(meta_df, :slug => ByRow(isequal(slug_text)))
    
    if nrow(target_row) == 0
        println("❌ Error: Slug '$slug_text' not found in metadata.")
        return
    end

    target_year = target_row.ggis_death_year[1]
    
    if ismissing(target_year)
        println("❌ Error: ggis_death_year is missing for '$slug_text'.")
        return
    end

    println("\n" * "="^40)
    println("INSPECTION: $slug_text")
    println("="^40)
    println("Target (Death) Year: ", target_year)

    # 2. Check data availability in main timeseries
    year_mask = df.ident_year .== target_year
    # Ensure we don't error if slug_sym doesn't exist in df
    if !(slug_sym in propertynames(df))
        println("❌ Error: Column :$slug_sym not found in main DataFrame.")
        return
    end
    
    slug_mask = .!ismissing.(df[!, slug_sym])
    valid_rows = df[year_mask .& slug_mask, :]
    
    println("Rows found with data: ", nrow(valid_rows))

    # 3. Regional distribution and Denominators
    if nrow(valid_rows) > 0
        # Distribution in the data
        counts = combine(groupby(valid_rows, :ggis_region), nrow => :count)
        println("\n--- Regional Distribution (Numerators) ---")
        println(counts)

        # Matching denominators from the universe
        found_regions = counts.ggis_region
        denoms = filter(r -> r.ident_year == target_year && r.ggis_region in found_regions, regional_universe)
        
        println("\n--- Regional Universe (Denominators) for $target_year ---")
        println(denoms)
        
        # Calculate penetration on the fly for verification
        println("\n--- Calculated Penetrations ---")
        for row in eachrow(counts)
            d_row = filter(r -> r.ggis_region == row.ggis_region, denoms)
            if !isempty(d_row)
                d = d_row.total_countries_in_region[1]
                p = round(row.count / d, digits=3)
                println("Region $(row.ggis_region): $(row.count) / $d = $p")
            end
        end
    else
        println("⚠️ ALERT: No data found for this slug in its recorded death year.")
    end
    println("="^40 * "\n")
end

inspect_slug_geo

In [20]:
unique(meta_plus2.ggis_geo_classification)

4-element Vector{Union{Missing, Symbol}}:
 :other
 :regional
 :global
 missing

In [24]:
"""
Performs a geographic audit at the 'Peak Year' for each slug 
currently classified as :other.
"""
function peak_year_geo_audit(df, meta_df, regional_universe)
    # 1. Filter for slugs currently not classified as regional
    other_slugs = subset(meta_df, :ggis_geo_classification => ByRow(isequal(:other))).slug
    
    results = DataFrame(
        slug = String[],
        prefix = String[],
        peak_year = Int[],
        peak_count = Int[],
        max_regional_pen = Float64[]
    )

    println("Auditing $(length(other_slugs)) slugs at their peak...")

    for slug in other_slugs
        slug_sym = Symbol(slug)
        !(slug_sym in propertynames(df)) && continue
        
        # Find the year with the maximum number of non-missing observations
        counts_by_year = combine(groupby(df, :ident_year), 
            slug_sym => (x -> sum(.!ismissing.(x))) => :count)
        
        max_idx = argmax(counts_by_year.count)
        peak_year = counts_by_year.ident_year[max_idx]
        peak_count = counts_by_year.count[max_idx]
        
        peak_count == 0 && continue

        # Calculate regional penetrations at THIS peak year
        regional_pens = compute_regional_penetrations(df, slug_sym, peak_year, regional_universe)
        max_pen = maximum(regional_pens)
        
        prefix = subset(meta_df, :slug => ByRow(isequal(slug))).prefix[1]
        
        push!(results, (slug, prefix, peak_year, peak_count, max_pen))
    end

    # Group by prefix to see which ones hit the 0.80 threshold at their peak
    summary = combine(groupby(results, :prefix), 
        :max_regional_pen => maximum => :best_regional_reach)
    
    return sort!(summary, :best_regional_reach, rev=true)
end

# Run the audit
peak_audit = peak_year_geo_audit(df, meta_plus2, regional_universe)

# Display prefixes that reach at least 80% at their peak
println("\n>>> Top Potential Regional Prefixes (at Peak Year)")
display(peak_audit[peak_audit.best_regional_reach .>= 0.80, :])

Auditing 1248 slugs at their peak...

>>> Top Potential Regional Prefixes (at Peak Year)


Row,prefix,best_regional_reach
,String,Float64
1,aid,1.0
2,bl,1.0
3,br,1.0
4,bti,1.0
5,ciri,1.0
6,fao,1.0
7,fe,1.0
8,gendip,1.0
9,gggi,1.0


In [21]:

meta_plus2[isequal.(meta_plus2.ggis_geo_classification, :regional), [:slug, :ggis_geo_classification, :ggis_region_penetration]]

Row,slug,ggis_geo_classification,ggis_region_penetration
,String31,Symbol?,Array…?
1,aii_acc,regional,"[0.0, 0.0, 0.25, 0.979592, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
2,aii_aio,regional,"[0.0, 0.0, 0.25, 0.979592, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
3,aii_cilser,regional,"[0.0, 0.0, 0.25, 0.979592, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
4,aii_elec,regional,"[0.0, 0.0, 0.25, 0.979592, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
5,aii_pubm,regional,"[0.0, 0.0, 0.25, 0.979592, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
6,aii_q01,regional,"[0.0, 0.0, 0.25, 0.979592, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
7,aii_q02,regional,"[0.0, 0.0, 0.25, 0.979592, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
8,aii_q03,regional,"[0.0, 0.0, 0.25, 0.979592, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
9,aii_q04,regional,"[0.0, 0.0, 0.25, 0.979592, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"


In [18]:
meta_plus2[coalesce.(meta_plus2.ggis_geo_classification .== "regional", false), [:slug, :ggis_geo_classification, :ggis_region_penetration]]

Row,slug,ggis_geo_classification,ggis_region_penetration
,String31,Symbol?,Array…?


In [19]:
meta_plus2[isequal.(meta_plus2.ggis_geo_classification, "regional"), [:slug, :ggis_geo_classification, :ggis_region_penetration]]

Row,slug,ggis_geo_classification,ggis_region_penetration
,String31,Symbol?,Array…?


In [28]:
slug = "aii_aio"
inspect_slug_geo(df, meta_df, slug, regional_universe)


INSPECTION: aii_aio
Target (Death) Year: 2023
Rows found with data: 53

--- Regional Distribution (Numerators) ---
2×2 DataFrame
 Row │ ggis_region  count 
     │ Int64?       Int64 
─────┼────────────────────
   1 │           3      5
   2 │           4     48

--- Regional Universe (Denominators) for 2023 ---
2×3 DataFrame
 Row │ ident_year  ggis_region  total_countries_in_region 
     │ Int64       Int64?       Int64                     
─────┼────────────────────────────────────────────────────
   1 │       2023            3                         20
   2 │       2023            4                         49

--- Calculated Penetrations ---
Region 3: 5 / 20 = 0.25
Region 4: 48 / 49 = 0.98



In [29]:

slug = "iiag_he"
inspect_slug_geo(df, meta_df, slug, regional_universe)


INSPECTION: iiag_he
Target (Death) Year: 2023
Rows found with data: 53

--- Regional Distribution (Numerators) ---
2×2 DataFrame
 Row │ ggis_region  count 
     │ Int64?       Int64 
─────┼────────────────────
   1 │           3      5
   2 │           4     48

--- Regional Universe (Denominators) for 2023 ---
2×3 DataFrame
 Row │ ident_year  ggis_region  total_countries_in_region 
     │ Int64       Int64?       Int64                     
─────┼────────────────────────────────────────────────────
   1 │       2023            3                         20
   2 │       2023            4                         49

--- Calculated Penetrations ---
Region 3: 5 / 20 = 0.25
Region 4: 48 / 49 = 0.98



In [11]:
meta_plus2 = enrich_metadata_with_geographic_coverage(meta_plus1);


>>> Computing Geographic Coverage (Step 8)
    Global Threshold:   ≥ 0.95
    Regional Threshold: ≥ 0.8 (Core) AND ≤ 0.05 (Exclusion in 7+ regions)
    Pre-computing geographic universes...

>>> Geographic Classification Summary
    :other — 1248
    :global — 616
    :regional — 145


In [23]:
CSV.write("data/qog_metadata_plus2.csv", meta_plus2)

"data/qog_metadata_plus2.csv"

In [10]:
# is_global(x) =
#     !ismissing(x) &&
#     lowercase(strip(string(x))) == "global"

# starts_wdi_gdp(x) =
#     !ismissing(x) &&
#     startswith(string(x), "wdi_gdp")

# df2 = meta_plus2 |>
#     x -> subset(
#         x,
#         :slug => ByRow(starts_wdi_gdp),
#         :ggis_geo_classification => ByRow(is_global)
#     ) |>
#     x -> select(
#         x,
#         Not([:prefix, :description, :type, :provenance, :min_year, :max_year])
#     )


In [11]:
# select(
#     subset(
#         meta_plus2,
#         :slug => ByRow(s -> startswith(s, "wdi_gdp")),   # allow missing-handling by skipmissing
#         :ggis_geo_classification => ByRow(==("global"));
#         skipmissing=true
#     ),
#     Not([:label, :prefix, :description, :type, :provenance, :min_year, :max_year])
# )

In [12]:
# select(
#     subset(
#         meta_plus2,
#         :slug => ByRow(s -> !ismissing(s) && startswith(s, "wdi_gdp")),
#         :ggis_geo_classification => ByRow(g -> !ismissing(g) && g == "global"),
#     ),
#     Not([:label, :prefix, :description, :type, :provenance, :min_year, :max_year])
# )

In [13]:
# select(
#     filter(:slug => s -> startswith(s, "wdi_gdp"), meta_plus2),
#     Not(:label, :prefix, :description, :type, :provenance, :min_year, :max_year)
# )

In [6]:
run_enrich_metadata_samples();


  enrich_metadata.jl — Function intent and usage

┌─ load_dataframes()
│  INTENT: Load the main QoG timeseries and the joined metadata in one call.
│  USE WHEN: You need both df and meta_df for auditing or enrichment.
│
│  RETURNS: (df, meta_df)
│    - df: main timeseries from load_qog_timeseries()
│    - meta_df: from PATH_METADATA_JOINED
│
│  USAGE:
│    df, meta = load_dataframes()
└──────────────────────────────────────────────────────────────────────────

┌─ classify_temporal_profile(birth_year, death_year; kwargs...)
│  INTENT: Classify a variable's temporal profile from its lifespan.
│  USE WHEN: You have first/last year and want :anchor, :experimental,
│           :legacy, :historical, :current, :modern, or :unclassified.
│
│  ARGUMENTS:
│    birth_year::Int, death_year::Int (positional)
│    Optional kwargs: data_start, data_end, current_year, active_lag, thresholds
│
│  RETURNS: Symbol (e.g. :anchor, :current)
│
│  USAGE:
│    profile = classify_temporal_profile(1946, 2022)
